# Defense Score Variants: Summed z-score, log2 z-score, singscore

Compares three ways of turning the 148 load-correlated defense genes (`load_correlated_genes_union_GO_and_FLG_plates_1_to_5.csv`) into one score per sample:

1. **Current:** z-score of TPM for each gene across samples, summed and divided by the number of genes.
2. **log2:** the same, but on log2(TPM + 1). Less driven by a few samples with extreme expression.
3. **singscore** (Foroutan et al. 2018), computed with the PySingscore package (`singscore`, installed from github.com/DavisLaboratory/PySingscore) using `norm_method="theoretical"`, the normalisation from the paper. Within each sample, all expressed genes are ranked by expression. The score is the mean rank of the defense genes, scaled so −0.5 means they are all at the bottom and +0.5 all at the top. Because it's computed within each sample, it's insensitive to global shifts such as library size.

PySingscore calls `DataFrame.append`, which was removed in pandas 2.0. The first code cell restores it as a thin wrapper around `pd.concat` so the package runs unchanged. Only defense genes present in the ranked gene set are passed to it: the package mishandles missing signature genes by shrinking the signature length cumulatively across samples.

There are no usable technical replicates (the re-run samples were re-run because of very low mapped reads), so the scores are compared on:
- **split-half reliability:** split the defense genes randomly in half many times and correlate the two half-scores (Spearman–Brown corrected). This measures internal consistency.
- **library size:** correlation with log10 total reads within timepoint; lower is better.
- **known markers and total load:** correlation within timepoint, as a sanity check.
- **distribution and outliers, agreement between scores, seasonal trend,** and overlap of the highest-scoring samples.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import zscore, spearmanr, skew

# Compatibility shim: PySingscore uses DataFrame.append, removed in pandas 2.0
if not hasattr(pd.DataFrame, "append"):
    pd.DataFrame.append = lambda self, other, ignore_index=False: pd.concat(
        [self, other], ignore_index=ignore_index
    )
from singscore.singscore import score as singscore_score

###Define Thresholds
singscore_min_fraction_expressed = (
    0.5  # genes ranked: TPM > 0 in at least this fraction of samples
)
n_split_halves = 100
top_n_samples = 25
random_seed = 0

In [2]:
transcriptome = pd.read_csv(
    "/Users/michael/Data/Luke_terrace_experiment/General_data/plate1_5_tpm_normalized.csv",
    index_col=0,
)
rows_to_drop_expression_data = [
    "A2450525897_n01_undetermined",
    "A2449446903_n01_undetermined",
    "B250508004_n01_undetermined",
    "B2449500127_n01_undetermined",
]
transcriptome = transcriptome.drop(index=rows_to_drop_expression_data)
transcriptome = transcriptome.sort_index()
metadata = pd.read_csv(
    "/Users/michael/Data/Luke_terrace_experiment/General_data/Plates_1_to_5_metadata_merged_luke.csv",
    index_col=0,
)
## Get plate 5 samples and remove what they replaced from the metadata to avoid duplicates
plate_5_replacements = metadata.loc[metadata["rnaprepplate"] == "LICRNA_05"]
non_plate_5_samples = metadata.loc[metadata["rnaprepplate"] != "LICRNA_05"]
replaced_samples = non_plate_5_samples.loc[
    non_plate_5_samples["sampID"].isin(plate_5_replacements["sampID"])
]
trimmed_metadata = metadata.drop(index=replaced_samples.index)
### For remaining duplicated sampIDs, drop the transcriptome with fewer total reads
double_duplicates = (
    trimmed_metadata.loc[trimmed_metadata.duplicated(subset="sampID", keep=False)]
    .sort_values(by="Total Reads")
    .drop_duplicates(subset="sampID", keep="first")
)
trimmed_metadata = trimmed_metadata.drop(index=double_duplicates.index)
### Drop failed libraries with too few reads (e.g. LIC162 has 66 total reads)
min_total_reads = 250_000
low_read_samples = trimmed_metadata.index[
    trimmed_metadata["Total Reads"] < min_total_reads
]
trimmed_metadata = trimmed_metadata.drop(index=low_read_samples)
print(
    f"Dropped {len(low_read_samples)} transcriptomes with < {min_total_reads:,} total reads"
)
trimmed_transcriptome = transcriptome.loc[trimmed_metadata.index]
print("Transcriptomes:", len(trimmed_metadata))

defense_gene_list = pd.read_csv(
    "/Users/michael/Data/Luke_terrace_experiment/General_data/Generated_data/load_correlated_genes_union_GO_and_FLG_plates_1_to_5.csv"
)
defense_genes = [
    g for g in defense_gene_list["gene"] if g in trimmed_transcriptome.columns
]
print("Defense genes in transcriptome:", len(defense_genes))

Dropped 6 transcriptomes with < 250,000 total reads
Transcriptomes: 426
Defense genes in transcriptome: 148


## Scores

In [3]:
def summed_z_score(expression, genes):
    return expression[genes].apply(zscore).sum(axis=1) / len(genes)


def singscore(expression, genes):
    """singscore from the PySingscore package (expects genes as rows, samples as columns)."""
    result = singscore_score(
        up_gene=list(genes), sample=expression.T, norm_method="theoretical"
    )
    return result["total_score"].reindex(expression.index)


# Genes ranked by singscore: expressed (TPM > 0) in enough samples
expressed = trimmed_transcriptome.columns[
    (trimmed_transcriptome > 0).mean() >= singscore_min_fraction_expressed
]
singscore_genes = [g for g in defense_genes if g in expressed]
print(
    f"Genes ranked for singscore: {len(expressed)}; defense genes among them: {len(singscore_genes)} of {len(defense_genes)}"
)

log_transcriptome = np.log2(trimmed_transcriptome + 1)
scores = pd.DataFrame(
    {
        "Current (z of TPM)": summed_z_score(trimmed_transcriptome, defense_genes),
        "log2 (z of log2 TPM+1)": summed_z_score(log_transcriptome, defense_genes),
        "singscore": singscore(trimmed_transcriptome[expressed], singscore_genes),
    }
)
score_names = scores.columns.tolist()
scores.describe().round(3)

Genes ranked for singscore: 18313; defense genes among them: 147 of 148


,Current (z of TPM),log2 (z of log2 TPM+1),singscore
count,426.000,426.000,426.000
mean,0.000,0.000,0.139
std,0.605,0.572,0.032
min,-0.825,-1.242,0.070
25%,-0.396,-0.417,0.117
50%,-0.143,-0.077,0.134
75%,0.222,0.337,0.157
max,5.238,2.759,0.314


## Distributions and outliers

In [4]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
fig.patch.set_facecolor("white")
for ax, name in zip(axes, score_names):
    sns.histplot(scores[name], bins=40, color="black", ax=ax)
    standardised = (scores[name] - scores[name].mean()) / scores[name].std()
    ax.set_title(
        f"{name}: skew {skew(scores[name]):.2f}, max {standardised.max():.1f} SD above mean",
        fontsize=10,
    )
    ax.set_xlabel(name)
    sns.despine(ax=ax)
plt.tight_layout()

## Agreement between scores

In [5]:
display(scores.corr(method="spearman").round(3))
g = sns.pairplot(scores, plot_kws={"s": 12, "alpha": 0.5}, corner=True, height=2.8)

# Overlap of the highest-scoring samples
top_sets = {
    name: set(scores[name].nlargest(top_n_samples).index) for name in score_names
}
overlap = pd.DataFrame(
    [[len(top_sets[a] & top_sets[b]) for b in score_names] for a in score_names],
    index=score_names,
    columns=score_names,
)
print(f"Overlap of the top {top_n_samples} samples:")
overlap

,Current (z of TPM),log2 (z of log2 TPM+1),singscore
Current (z of TPM),1.000,0.991,0.756
log2 (z of log2 TPM+1),0.991,1.000,0.786
singscore,0.756,0.786,1.000


Overlap of the top 25 samples:


,Current (z of TPM),log2 (z of log2 TPM+1),singscore
Current (z of TPM),25,22,15
log2 (z of log2 TPM+1),22,25,15
singscore,15,15,25


## Split-half reliability

The defense genes are split randomly in half `n_split_halves` times. Each half gives its own score, and the correlation between the two halves (Spearman–Brown corrected to full length) measures how consistently the genes agree on a sample's score. Computed across all samples, and within timepoint (timepoint means removed from both halves).

In [6]:
rng = np.random.default_rng(random_seed)
timepoints = trimmed_metadata["timepoint"]
score_functions = {
    "Current (z of TPM)": lambda genes: summed_z_score(trimmed_transcriptome, genes),
    "log2 (z of log2 TPM+1)": lambda genes: summed_z_score(log_transcriptome, genes),
    "singscore": lambda genes: singscore(trimmed_transcriptome[expressed], genes),
}
remove_timepoint_mean = lambda values: values - values.groupby(timepoints).transform(
    "mean"
)

reliability_rows = []
for _ in range(n_split_halves):
    shuffled = rng.permutation(singscore_genes)  # same gene pool for every method
    half_a, half_b = list(shuffled[: len(shuffled) // 2]), list(
        shuffled[len(shuffled) // 2 :]
    )
    for name, function in score_functions.items():
        a, b = function(half_a), function(half_b)
        for scope, (x, y) in {
            "all samples": (a, b),
            "within timepoint": (remove_timepoint_mean(a), remove_timepoint_mean(b)),
        }.items():
            r = np.corrcoef(x, y)[0, 1]
            reliability_rows.append(
                {
                    "score": name,
                    "scope": scope,
                    "split-half r": r,
                    "Spearman-Brown": 2 * r / (1 + r),
                }
            )
reliability = pd.DataFrame(reliability_rows)
reliability_summary = (
    reliability.groupby(["score", "scope"])["Spearman-Brown"]
    .agg(["median", "min", "max"])
    .round(3)
)
display(reliability_summary)

fig, ax = plt.subplots(figsize=(8, 4.5))
fig.patch.set_facecolor("white")
sns.boxplot(data=reliability, x="score", y="Spearman-Brown", hue="scope", ax=ax)
ax.set_xlabel("")
ax.set_ylabel("Split-half reliability (Spearman-Brown)")
sns.despine(ax=ax)
plt.tight_layout()

median    min    max
score                  scope                                 
Current (z of TPM)     all samples        0.989  0.980  0.993
                       within timepoint   0.984  0.966  0.990
log2 (z of log2 TPM+1) all samples        0.987  0.975  0.992
                       within timepoint   0.983  0.972  0.989
singscore              all samples        0.933  0.896  0.949
                       within timepoint   0.916  0.868  0.945

## Library size, total load and known markers (within timepoint)

Spearman correlation after removing timepoint means. For library size, closer to 0 is better. Total load and markers are sanity checks: the genes were picked for correlating with load, so all scores should track it. Markers that are themselves in the 148-gene set are flagged, since their correlation is partly built in.

In [7]:
total_load = pd.read_csv(
    "/Users/michael/Data/Luke_terrace_experiment/Microbiome/lic2024_ham_16S_ITS.csv",
    index_col=0,
)["ham16s"]
covariates = pd.DataFrame(
    {
        "log10 total reads": np.log10(trimmed_metadata["Total Reads"].astype(float)),
        "log10 total load (ham16s)": np.log10(
            trimmed_metadata["sampID"].map(total_load)
        ),
    }
)
marker_genes = {
    "PR1": "AT2G14610",
    "PR2": "AT3G57260",
    "PR5": "AT1G75040",
    "PAD4": "AT3G52430",
    "WRKY33": "AT2G38470",
    "FRK1": "AT2G19190",
    "PAD3": "AT3G26830",
    "CYP71A13": "AT2G30770",
    "NHL10": "AT2G35980",
    "PDF1.2": "AT5G44420",
    "JAZ1": "AT1G19180",
}
for marker, gene in marker_genes.items():
    if gene in log_transcriptome.columns:
        label = f"{marker} ({'in set' if gene in defense_genes else 'not in set'})"
        covariates[label] = log_transcriptome[gene]

rows = {}
for name in score_names:
    score_within = remove_timepoint_mean(scores[name])
    rows[name] = {}
    for covariate in covariates.columns:
        ok = covariates[covariate].notna()
        rows[name][covariate] = spearmanr(
            score_within[ok], remove_timepoint_mean(covariates[covariate])[ok]
        )[0]
within_correlations = pd.DataFrame(rows).round(3)
within_correlations

,Current (z of TPM),log2 (z of log2 TPM+1),singscore
log10 total reads,-0.231,-0.135,-0.076
log10 total load (ham16s),0.239,0.231,0.241
PR1 (not in set),0.415,0.444,0.611
PR2 (in set),0.596,0.624,0.649
PR5 (not in set),0.562,0.598,0.606
PAD4 (in set),0.739,0.747,0.713
WRKY33 (not in set),0.530,0.547,0.587
FRK1 (not in set),0.437,0.449,0.542
PAD3 (not in set),0.442,0.470,0.542
CYP71A13 (not in set),0.346,0.379,0.427


In [8]:
fig, ax = plt.subplots(figsize=(7, 0.45 * len(within_correlations) + 1.5))
fig.patch.set_facecolor("white")
sns.heatmap(
    within_correlations,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    vmin=-1,
    vmax=1,
    ax=ax,
    cbar_kws={"label": "Spearman r, within timepoint"},
)
ax.set_title("Score vs library size, load and markers (within timepoint)")
plt.tight_layout()

## Seasonal trend

In [9]:
trend = scores.apply(lambda s: (s - s.mean()) / s.std())
trend["Date and Time"] = pd.to_datetime(
    trimmed_metadata["date"] + " " + trimmed_metadata["time"], format="%m/%d/%y %H:%M"
)
trend["Seasonal_or_circadian"] = np.where(
    trimmed_metadata["Experiment Type"] == "Long Term", "Seasonal", "Circadian"
)
trend_long = trend.loc[trend["Seasonal_or_circadian"] == "Seasonal"].melt(
    id_vars=["Date and Time", "Seasonal_or_circadian"],
    value_vars=score_names,
    var_name="score",
    value_name="standardised score",
)

fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor("white")
sns.lineplot(
    data=trend_long,
    x="Date and Time",
    y="standardised score",
    hue="score",
    errorbar=("ci", 95),
    ax=ax,
)
ax.set_ylabel("Score (standardised), seasonal samples")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=45)
sns.despine(ax=ax)
plt.tight_layout()

In [10]:
scores.to_csv(
    "/Users/michael/Data/Luke_terrace_experiment/General_data/Generated_data/defense_score_variants.csv"
)

In [12]:
tpm_scores = scores[["log2 (z of log2 TPM+1)"]]

In [13]:
tpm_scores

,log2 (z of log2 TPM+1)
filename,
A2450525897_n01_LICRNA01_A01,1.365245
A2450525897_n01_LICRNA01_B01,0.969967
A2450525897_n01_LICRNA01_C01,1.882054
A2450525897_n01_LICRNA01_E01,0.700467
A2450525897_n01_LICRNA01_F01,2.005185
...,...
A2534491401_n01_LICRNA05_D12,0.042505
A2534491401_n01_LICRNA05_E12,-0.690517
A2534491401_n01_LICRNA05_F12,0.534769


In [14]:
tpm_scores.to_csv(
    "/Users/michael/Data/Luke_terrace_experiment/General_data/Generated_data/tpm_defense_score_mod.csv"
)